In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

In [32]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (569, 30)
Shape y: (569,)


In [33]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [34]:
# Logistic Regression
log = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000))
])

log.fit(X_train, y_train)
acc_log = accuracy_score(y_test, log.predict(X_test))
print("Logistic (No PCA):", acc_log)

# KNN
knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5))
])

knn.fit(X_train, y_train)
acc_knn = accuracy_score(y_test, knn.predict(X_test))
print("KNN (No PCA):", acc_knn)

# SVM RBF
svm = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf'))
])

svm.fit(X_train, y_train)
acc_svm = accuracy_score(y_test, svm.predict(X_test))
print("SVM (No PCA):", acc_svm)

Logistic (No PCA): 0.9736842105263158
KNN (No PCA): 0.9473684210526315
SVM (No PCA): 0.9824561403508771


In [35]:
pca = PCA(n_components=10)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [36]:
log_pca = LogisticRegression(max_iter=5000)

log_pca.fit(X_train_pca, y_train)

acc_log_pca = accuracy_score(
    y_test,
    log_pca.predict(X_test_pca)
)

print("Logistic + PCA:", acc_log_pca)

Logistic + PCA: 0.9824561403508771


In [43]:
knn_pca = KNeighborsClassifier(n_neighbors=5)

knn_pca.fit(X_train_pca, y_train)

acc_knn_pca = accuracy_score(
    y_test,
    knn_pca.predict(X_test_pca)
)

print("KNN + PCA:", acc_knn_pca)

KNN + PCA: 0.956140350877193


In [44]:
svm_pca = SVC(kernel='rbf')

svm_pca.fit(X_train_pca, y_train)

acc_svm_pca = accuracy_score(
    y_test,
    svm_pca.predict(X_test_pca)
)

print("SVM + PCA:", acc_svm_pca)

SVM + PCA: 0.9649122807017544


In [37]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN', 'SVM RBF'],
    'Tanpa PCA': [acc_log, acc_knn, acc_svm],
    'Dengan PCA (10 Komponen)': [acc_log_pca, acc_knn_pca, acc_svm_pca]
})

results

,Model,Tanpa PCA,Dengan PCA (10 Komponen)
0,Logistic Regression,0.973684,0.982456
1,KNN,0.947368,0.956140
2,SVM RBF,0.982456,0.964912


In [41]:
cv_log = cross_val_score(
    log,
    X_train_scaled,
    y_train,
    cv=5
).mean()

cv_log_pca = cross_val_score(
    log_pca,
    X_train_pca,
    y_train,
    cv=5
).mean()

print("CV Logistic:", cv_log)
print("CV Logistic + PCA:", cv_log_pca)

CV Logistic: 0.9758241758241759
CV Logistic + PCA: 0.9758241758241759


In [42]:
component_settings = {
    "PCA 5": 5,
    "PCA 10": 10,
    "PCA 15": 15,
    "PCA 90%": 0.90
}

experiment_results = []

for label, comp in component_settings.items():

    pca = PCA(n_components=comp)

    X_train_temp = pca.fit_transform(X_train_scaled)
    X_test_temp = pca.transform(X_test_scaled)

    # Logistic
    log = LogisticRegression(max_iter=5000)
    log.fit(X_train_temp, y_train)

    # KNN
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_temp, y_train)

    # SVM
    svm = SVC(kernel='rbf')
    svm.fit(X_train_temp, y_train)

    experiment_results.append([
        label,
        accuracy_score(y_test, log.predict(X_test_temp)),
        accuracy_score(y_test, knn.predict(X_test_temp)),
        accuracy_score(y_test, svm.predict(X_test_temp))
    ])

experiment_df = pd.DataFrame(
    experiment_results,
    columns=[
        "PCA Setting",
        "Logistic",
        "KNN",
        "SVM"
    ]
)

experiment_df

,PCA Setting,Logistic,KNN,SVM
0,PCA 5,0.982456,0.956140,0.964912
1,PCA 10,0.982456,0.956140,0.964912
2,PCA 15,0.991228,0.947368,0.982456
3,PCA 90%,0.982456,0.947368,0.973684
